In [16]:
from datasets import load_dataset

common_sense_qa_dataset = load_dataset('tau/commonsense_qa')
common_sense_qa_dataset

DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
        num_rows: 1140
    })
})

In [17]:
common_sense_qa_dataset['train'][0]

{'id': '075e483d21c29a511267ef62bedc0461',
 'question': 'The sanctions against the school were a punishing blow, and they seemed to what the efforts the school had made to change?',
 'question_concept': 'punishing',
 'choices': {'label': ['A', 'B', 'C', 'D', 'E'],
  'text': ['ignore', 'enforce', 'authoritarian', 'yell at', 'avoid']},
 'answerKey': 'A'}

In [18]:
from transformers import AutoTokenizer

# Load the tokenizer for a specific Gemma model
MODEL_ID = "google/gemma-4-E2B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

In [19]:
tokenizer.special_tokens_map

{'bos_token': '<bos>',
 'eos_token': '<eos>',
 'unk_token': '<unk>',
 'pad_token': '<pad>',
 'mask_token': '<mask>',
 'audio_token': '<|audio|>',
 'boa_token': '<|audio>',
 'boi_token': '<|image>',
 'eoa_token': '<audio|>',
 'eoc_token': '<channel|>',
 'eoi_token': '<image|>',
 'eot_token': '<turn|>',
 'escape_token': '<|"|>',
 'etc_token': '<tool_call|>',
 'etd_token': '<tool|>',
 'etr_token': '<tool_response|>',
 'image_token': '<|image|>',
 'soc_token': '<|channel>',
 'sot_token': '<|turn>',
 'stc_token': '<|tool_call>',
 'std_token': '<|tool>',
 'str_token': '<|tool_response>',
 'think_token': '<|think|>'}

In [20]:
tokenizer.encode(tokenizer.special_tokens_map['pad_token'])

[2, 0]

In [21]:
tokenizer.encode("A.")

[2, 236776, 236761]

In [22]:
def sanity_checks(dataset_rows):
    for question, choices, answerKey in zip(dataset_rows['question'], dataset_rows['choices'], dataset_rows['answerKey']):
        if len(choices['label']) != 5:
            raise ValueError(f"Expected 5 choices, got {len(choices['label'])}")
        if answerKey not in choices['label']:
            raise ValueError(f"Answer key '{answerKey}' not found in choices {choices['label']}")
        if len(choices['text']) != len(choices['label']):
            raise ValueError(f"Number of choice texts {len(choices['text'])} does not match number of labels {len(choices['label'])}")
    print("All sanity checks passed!")

In [23]:
sanity_checks(common_sense_qa_dataset['train'])

All sanity checks passed!


In [101]:
def prepare_dataset(dataset_rows):
    prompts = []
    for question, choices, answerKey in zip(dataset_rows['question'], dataset_rows['choices'], dataset_rows['answerKey']):
        prompt = f'<question> {question} \n' + f'<options>\n'
        for label, text in zip(choices['label'], choices['text']):
            prompt = prompt + f'{label}. {text}\n'
        prompt = prompt + f'{tokenizer.special_tokens_map["eos_token"]}'
        prompts.append((prompt, answerKey.strip()))
    
    return prompts

In [102]:
prompts = prepare_dataset(common_sense_qa_dataset['train'])

In [103]:
print(prompts[8004][0])
print(f'Label: {prompts[8004][1]}')

<question> The cow farm was free range, it provided all the quality meat to t he bay area of where? 
<options>
A. slaughter house
B. rural area
C. nursery rhyme
D. northern california
E. farmyard
<eos>
Label: D


In [104]:
def tokenize_dataset(prompts):
    tokenized_prompts = tokenizer.encode(prompts[0])
    return tokenized_prompts, prompts[1]

In [105]:
from joblib import Parallel, delayed
from tqdm.auto import tqdm

tasks = (delayed(tokenize_dataset)(pair) for pair in prompts)

# Run in parallel
tokenized_prompts = list(tqdm(
    Parallel(n_jobs=-1, return_as="generator", batch_size=500)(tasks), 
    total=len(prompts),
    desc="Tokenizing"
))

Tokenizing: 100%|██████████| 9741/9741 [01:03<00:00, 153.57it/s] 


In [108]:
len(tokenized_prompts), len(tokenized_prompts[0]), len(tokenized_prompts[0][0]), tokenized_prompts[0][1]

(9741, 2, 56, 'A')

In [109]:
from datasets import Dataset

def prepare_tokenized_dataset(tokenized_prompts):
    tokenized_dataset_dict = {
        'input_ids': [],
        'labels': []
    }
    for tokenized_prompt in tokenized_prompts:
        tokenized_dataset_dict['input_ids'].append(tokenized_prompt[0])
        tokenized_dataset_dict['labels'].append(tokenized_prompt[1])
    return Dataset.from_dict(tokenized_dataset_dict)

In [110]:
tokenized_dataset = prepare_tokenized_dataset(tokenized_prompts)

In [111]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 9741
})

In [114]:
print(tokenized_dataset['input_ids'][100])
print(tokenized_dataset['labels'][100])

[2, 236820, 15884, 236813, 669, 72467, 1456, 496, 25805, 14578, 236764, 668, 7953, 886, 657, 2033, 532, 886, 657, 506, 1144, 236881, 236743, 107, 236820, 8194, 236813, 107, 236776, 236761, 15348, 40682, 107, 236799, 236761, 21116, 107, 236780, 236761, 4408, 107, 236796, 236761, 123683, 107, 236788, 236761, 3207, 11967, 107, 1]
C


In [115]:
validation_prompts = prepare_dataset(common_sense_qa_dataset['validation'])
test_prompts = prepare_dataset(common_sense_qa_dataset['test'])

In [116]:
tasks = (delayed(tokenize_dataset)(pair) for pair in validation_prompts)

# Run in parallel
validation_tokenized_prompts = list(tqdm(
    Parallel(n_jobs=-1, return_as="generator", batch_size=500)(tasks), 
    total=len(validation_prompts),
    desc="Tokenizing"
))

Tokenizing: 100%|██████████| 1221/1221 [01:00<00:00, 20.07it/s]


In [117]:
tasks = (delayed(tokenize_dataset)(pair) for pair in test_prompts)

# Run in parallel
test_tokenized_prompts = list(tqdm(
    Parallel(n_jobs=-1, return_as="generator", batch_size=100)(tasks), 
    total=len(test_prompts),
    desc="Tokenizing"
))

Tokenizing: 100%|██████████| 1140/1140 [00:57<00:00, 19.89it/s]


In [120]:
print(tokenizer.decode(validation_tokenized_prompts[301][0]))
print(f'Label: {validation_tokenized_prompts[301][1]}')

<bos><question> Donald is a prominent figure for the federal government, so in what city does he likely spend a lot of time? 
<options>
A. everything
B. capitol building
C. tourist sites
D. canada
E. washington d.c
<eos>
Label: E


In [121]:
validation_tokenized_dataset = prepare_tokenized_dataset(validation_tokenized_prompts)
test_tokenized_dataset = prepare_tokenized_dataset(test_tokenized_prompts)

In [122]:
tokenized_dataset, validation_tokenized_dataset, test_tokenized_dataset

(Dataset({
     features: ['input_ids', 'labels'],
     num_rows: 9741
 }),
 Dataset({
     features: ['input_ids', 'labels'],
     num_rows: 1221
 }),
 Dataset({
     features: ['input_ids', 'labels'],
     num_rows: 1140
 }))

In [123]:
from datasets import DatasetDict

final_dataset = DatasetDict({
    'train': tokenized_dataset,
    'validation': validation_tokenized_dataset,
    'test': test_tokenized_dataset
})

In [124]:
final_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['input_ids', 'labels'],
        num_rows: 1140
    })
})

In [125]:
final_dataset.save_to_disk('./datasets/common_sense_qa_tokenized_dataset')

Saving the dataset (1/1 shards): 100%|██████████| 1140/1140 [00:00<00:00, 487610.30 examples/s]


In [14]:
from datasets import load_from_disk

common_sense_qa_dataset = load_from_disk('./datasets/common_sense_qa_tokenized_dataset')
common_sense_qa_dataset

DatasetDict({
    train: Dataset({
        features: ['input_ids'],
        num_rows: 9741
    })
    validation: Dataset({
        features: ['input_ids'],
        num_rows: 1221
    })
    test: Dataset({
        features: ['input_ids'],
        num_rows: 1140
    })
})